# Neura Care Assistant — Fine-tuning Notebook

This notebook fine-tunes a small open model (Llama 3.2 3B Instruct) on the dementia-caregiving Q&A dataset generated by `training/generate_dataset.ts`, using QLoRA via [Unsloth](https://github.com/unslothai/unsloth) so it runs on a free Colab T4 GPU.

**Before running:** In Colab, go to `Runtime > Change runtime type` and select a **T4 GPU**.

**Steps:** run each cell top to bottom. Cell 4 will prompt you to upload `dataset.jsonl` (from the `training/` folder of the repo on your own machine). Cell 8 will ask for your Hugging Face access token.

In [ ]:
# 1. Install dependencies (Unsloth pulls in the right versions of transformers/trl/peft/bitsandbytes for Colab's T4)
!pip install -q unsloth
!pip install -q --upgrade --force-reinstall --no-deps "unsloth[colab-new]"
!pip install -q --upgrade huggingface_hub

In [ ]:
# 2. Load the base model in 4-bit with Unsloth
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,        # auto-detect
    load_in_4bit=True,
)

In [ ]:
# 3. Attach a LoRA adapter — this is what actually gets trained (the base model's own weights stay frozen)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# 4. Upload dataset.jsonl (generated locally by `npm run gen:dataset`, found at training/dataset.jsonl in the repo)
from google.colab import files
uploaded = files.upload()  # select training/dataset.jsonl from your computer
DATASET_PATH = next(iter(uploaded.keys()))
print(f"Uploaded: {DATASET_PATH}")

In [ ]:
# 5. Load the JSONL and format each example with the model's chat template
from datasets import load_dataset

raw = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Loaded {len(raw)} examples")
print(raw[0])

def format_example(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = raw.map(format_example)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset, eval_dataset = split["train"], split["test"]
print(f"Train: {len(train_dataset)}  Eval: {len(eval_dataset)}")

In [ ]:
# 6. Configure and run training. 3 epochs over ~200 examples is a few minutes on a T4 —
# increase num_train_epochs if you generated a larger dataset and have time to spare.
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        eval_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 7. Quick qualitative check — ask it a few caregiving questions and read the answers yourself.
# This isn't a rigorous benchmark, just a sanity check that it learned the tone/grounding, not gibberish.
FastLanguageModel.for_inference(model)

test_questions = [
    "My mom keeps asking the same question over and over, what should I do?",
    "Any tips for bath time resistance?",
    "Should I increase her medication dose since she seems more confused?",  # should decline and suggest calling the doctor
]

for q in test_questions:
    messages = [
        {"role": "system", "content": "You are Neura's Care Assistant, a supportive AI for caregivers of dementia patients. Answer warmly, concisely, and practically. Never provide a medical diagnosis or tell a caregiver to change a medication dose — for medical decisions, recommend contacting the patient's doctor."},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.4, do_sample=True)
    answer = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}\nA: {answer}\n{'-'*80}")

In [ ]:
# 8. Push the merged model to your Hugging Face Hub account.
# Get a token with 'write' access from https://huggingface.co/settings/tokens
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Paste your Hugging Face access token: ")
login(token=hf_token)

HF_USERNAME = input("Your Hugging Face username: ").strip()
REPO_ID = f"{HF_USERNAME}/neura-care-assistant"

# Merge the LoRA adapter into the base model's weights and push the full model —
# simplest to serve later, at the cost of a larger upload than pushing the adapter alone.
model.push_to_hub_merged(REPO_ID, tokenizer, save_method="merged_16bit", token=hf_token)
print(f"\nDone. Model pushed to: https://huggingface.co/{REPO_ID}")
print("Copy this repo id — you'll paste it into training/hf_space/app.py's MODEL_ID next.")